# WBCBench 2026 — Aşama 9A: ConvNeXt-Tiny Final Training — Seed 42

Bu notebook **final eğitim** içindir.

- Model: ConvNeXt-Tiny
- Pretraining: ImageNet
- Seed: 42
- Train: `phase2_train`
- Validation: `phase2_eval`
- Test: **KULLANILMAZ**
- Primary selection metric: Validation Macro-F1
- Maksimum epoch: 12
- Early stopping patience: 3
- Best full checkpoint + history CSV/JSON kaydedilir.

> ConvNeXt/ResNet için daha önce çalışan `medxai_gpu` kernelini kullanın; `Python (retfound)` kullanmayın.


In [1]:
from pathlib import Path
import random, time, json, csv
import numpy as np
import torch, torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:",torch.__version__)
print("Torchvision:",torchvision.__version__)
print("Device:",device)
if device.type=="cuda": print("GPU:",torch.cuda.get_device_name(0))
assert device.type=="cuda"


Torch: 2.11.0+cu126
Torchvision: 0.26.0+cu126
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
ROOT=Path.cwd()
TRAIN=ROOT/"wbc_project"/"data"/"processed"/"phase2_train"
VAL=ROOT/"wbc_project"/"data"/"processed"/"phase2_eval"
OUT=ROOT/"outputs"/"final"/"convnext_tiny"/"seed42"
OUT.mkdir(parents=True,exist_ok=True)

assert TRAIN.exists() and VAL.exists()
print("Output:",OUT.resolve())
print("TEST split bu notebook'ta tanımlanmıyor.")


Output: C:\Users\sarah\Desktop\scı5\outputs\final\convnext_tiny\seed42
TEST split bu notebook'ta tanımlanmıyor.


In [3]:
train_tf=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds=datasets.ImageFolder(TRAIN,transform=train_tf)
val_ds=datasets.ImageFolder(VAL,transform=val_tf)
assert train_ds.classes==val_ds.classes and len(train_ds.classes)==13

BATCH=8
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,
                        pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0,
                      pin_memory=True)

print("Classes:",train_ds.classes)
print("Train:",len(train_ds),"Validation:",len(val_ds),"Batch:",BATCH)


Classes: ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']
Train: 49794 Validation: 10700 Batch: 8


In [4]:
weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model=models.convnext_tiny(weights=weights)

# torchvision ConvNeXt classifier: [LayerNorm2d, Flatten, Linear]
in_features=model.classifier[2].in_features
model.classifier[2]=nn.Linear(in_features,13)
model=model.to(device)

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
scaler=torch.amp.GradScaler("cuda")

print("Parameters:",f"{sum(p.numel() for p in model.parameters()):,}")


Parameters: 27,830,125


In [5]:
def train_epoch():
    model.train()
    total_loss=0.0; n=0
    for i,(x,y) in enumerate(train_loader,1):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(x)
            loss=criterion(logits,y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss+=loss.item()*y.size(0); n+=y.size(0)
        if i%1000==0:
            print(f"  batch {i}/{len(train_loader)} | train_loss={total_loss/n:.4f}")
    return total_loss/n

@torch.inference_mode()
def validate():
    model.eval()
    total_loss=0.0; n=0; ys=[]; ps=[]
    for x,y in val_loader:
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(x); loss=criterion(logits,y)
        total_loss+=loss.item()*y.size(0); n+=y.size(0)
        ys.extend(y.cpu().tolist()); ps.extend(logits.argmax(1).cpu().tolist())
    return {
        "val_loss":total_loss/n,
        "macro_f1":f1_score(ys,ps,average="macro",zero_division=0),
        "balanced_acc":balanced_accuracy_score(ys,ps),
        "accuracy":accuracy_score(ys,ps)
    }


In [6]:
MAX_EPOCHS=12
PATIENCE=3
best_f1=-1.0
bad_epochs=0
history=[]
best_path=OUT/"convnext_tiny_seed42_best.pt"

config={
    "model":"convnext_tiny","seed":SEED,"max_epochs":MAX_EPOCHS,
    "patience":PATIENCE,"batch_size":BATCH,"lr":1e-4,"weight_decay":1e-4,
    "input_size":224,"selection_metric":"validation_macro_f1",
    "test_used":False,"classes":train_ds.classes
}
(OUT/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")

for epoch in range(1,MAX_EPOCHS+1):
    print("\n"+"="*72)
    print(f"CONVNEXT-TINY FINAL — SEED 42 — EPOCH {epoch}/{MAX_EPOCHS}")
    print("="*72)
    t=time.time()
    train_loss=train_epoch()
    m=validate()
    row={"epoch":epoch,"train_loss":train_loss,**m,"minutes":(time.time()-t)/60}
    history.append(row)

    print(f"train_loss={train_loss:.4f} | val_loss={m['val_loss']:.4f} | "
          f"Macro-F1={m['macro_f1']:.4f} | Balanced Acc={m['balanced_acc']:.4f} | "
          f"Acc={m['accuracy']:.4f} | {row['minutes']:.1f} dk")

    if m["macro_f1"]>best_f1:
        best_f1=m["macro_f1"]; bad_epochs=0
        torch.save({
            "model_state_dict":model.state_dict(),
            "classes":train_ds.classes,
            "seed":SEED,"epoch":epoch,
            "val_macro_f1":best_f1,
            "config":config
        },best_path)
        print("✓ Yeni best checkpoint:",best_path)
    else:
        bad_epochs+=1
        print(f"İyileşme yok: {bad_epochs}/{PATIENCE}")

    (OUT/"history.json").write_text(json.dumps(history,ensure_ascii=False,indent=2),encoding="utf-8")
    with open(OUT/"history.csv","w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)

    if bad_epochs>=PATIENCE:
        print("✓ Early stopping.")
        break

print("\n"+"="*72)
print("CONVNEXT-TINY FINAL TRAINING — SEED 42 SONUCU")
print("="*72)
print("Best validation Macro-F1:",f"{best_f1:.4f}")
print("Best checkpoint:",best_path)
print("Epochs completed:",len(history))
print("✓ TEST SPLIT KULLANILMADI.")



CONVNEXT-TINY FINAL — SEED 42 — EPOCH 1/12
  batch 1000/6225 | train_loss=0.4823
  batch 2000/6225 | train_loss=0.4052
  batch 3000/6225 | train_loss=0.3645
  batch 4000/6225 | train_loss=0.3384
  batch 5000/6225 | train_loss=0.3224
  batch 6000/6225 | train_loss=0.3086
train_loss=0.3063 | val_loss=0.2945 | Macro-F1=0.6148 | Balanced Acc=0.5908 | Acc=0.9164 | 15.3 dk
✓ Yeni best checkpoint: C:\Users\sarah\Desktop\scı5\outputs\final\convnext_tiny\seed42\convnext_tiny_seed42_best.pt

CONVNEXT-TINY FINAL — SEED 42 — EPOCH 2/12
  batch 1000/6225 | train_loss=0.2123
  batch 2000/6225 | train_loss=0.2072
  batch 3000/6225 | train_loss=0.2076
  batch 4000/6225 | train_loss=0.2064
  batch 5000/6225 | train_loss=0.2065
  batch 6000/6225 | train_loss=0.2032
train_loss=0.2032 | val_loss=0.2409 | Macro-F1=0.6075 | Balanced Acc=0.5968 | Acc=0.9282 | 10.7 dk
İyileşme yok: 1/3

CONVNEXT-TINY FINAL — SEED 42 — EPOCH 3/12
  batch 1000/6225 | train_loss=0.1561
  batch 2000/6225 | train_loss=0.1642
  ba